# LiDAR Utility Corridor — Full Analysis

End-to-end pipeline: raw LiDAR → ground classification → CHM → corridor exceedance detection → accuracy assessment.

**Sections**
1. Data Acquisition and Point Cloud Inspection
2. PDAL Processing: Ground Classification and Height Normalization
3. CHM Derivation: DSM, DTM, and Canopy Height Model
4. Corridor Analysis: Buffering, Clipping, and Exceedance Detection
5. Accuracy Assessment: Thomas Fire Perimeter Validation

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import laspy
import contextily as ctx
import rioxarray
import requests
from shapely.geometry import shape

ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
sys.path.append(str(ROOT))

from src.data_utils import load_laz, inspect_point_cloud, validate_crs
from src.pdal_runner import check_pdal_available, run_pipeline
from src.chm import rasterize_to_dsm, rasterize_to_dtm, compute_chm
from src.corridor import load_corridor_centerline, buffer_corridor, clip_chm_to_corridor, threshold_exceedance
# from validation.accuracy_assessment import assess_chm_accuracy  # implement in Part 5

raw_las_path          = ROOT / "data" / "raw" / "socal_sespe_32611.laz"
pipeline_json_path    = ROOT / "pipeline" / "pdal_pipeline.json"
normalized_las_path   = ROOT / "data" / "processed" / "socal_sespe_32611_normalized.laz"
dsm_path              = ROOT / "outputs" / "rasters" / "socal_sespe_32611_dsm.tif"
dtm_path              = ROOT / "outputs" / "rasters" / "socal_sespe_32611_dtm.tif"
chm_path              = ROOT / "outputs" / "rasters" / "socal_sespe_32611_chm.tif"
clipped_chm_path      = ROOT / "outputs" / "rasters" / "socal_sespe_32611_chm_clipped.tif"
fire_perimeter_path   = ROOT / "data" / "raw" / "thomas_fire_perimeter.gpkg"
accuracy_summary_path = ROOT / "outputs" / "tables" / "accuracy_summary.csv"

---

## Part 1 — Data Acquisition and Point Cloud Inspection

Loads the USGS 3DEP LiDAR tile for the North Fillmore / Sespe Creek study area and inspects its structure before any processing. No PDAL processing is performed here — diagnostic only.

**Inputs:** `data/raw/socal_sespe_32611.laz` — USGS 3DEP tile (SoCal Wildfires B3 2018), ~5.6M points, EPSG:32611

**Confirm before Part 2:**
- Point count ~5,578,060
- EPSG is 32611
- Estimated density ~5.2 pts/m²
- Classification distribution: Class 1 (unclassified ~3.3M), Class 2 (ground ~2.3M), Class 7 (noise), Class 18 (high noise)
- Class 2 already present — ground was pre-classified by USGS 3DEP

In [ ]:
las = load_laz(raw_las_path)

df = pd.DataFrame({
    "x": np.array(las.x).flatten(),
    "y": np.array(las.y).flatten(),
    "z": np.array(las.z).flatten(),
    "classification": las.classification,
    "return_number": las.return_number,
    "intensity": las.intensity,
})

print(df.describe())
df.head()

In [ ]:
inspect_point_cloud(las)

z range: 150 — 400 — that's elevation above ellipsoid, which makes sense for foothill terrain. Confirms the vertical datum is ellipsoidal as expected.

In [ ]:
validate_crs(las, expected_epsg=32611)

---

## Part 2 — PDAL Processing: Ground Classification and Height Normalization

Runs the PDAL pipeline on the raw point cloud. Two operations are applied:

1. **Ground classification** — `filters.csf` assigns ASPRS class 2 to ground returns
2. **Height normalization** — `filters.hag_nn` adds a `HeightAboveGround` dimension to every point

**Inputs:** `data/raw/socal_sespe_32611.laz`, `pipeline/pdal_pipeline.json`

**Outputs:** `data/processed/socal_sespe_32611_normalized.laz`

**Confirm before Part 3:**
- Output LAZ exists at `data/processed/socal_sespe_32611_normalized.laz`
- Re-inspect classification distribution — class 2 (ground) points should now be present
- If class 2 count is zero or suspiciously low, revisit CSF parameters (`resolution`, `rigidness`, `slope_smooth`)

**CSF parameter guidance** — terrain is chaparral foothill, sloped but not extreme:
- `resolution`: 0.5 or 1.0
- `rigidness`: 2 (default)
- `slope_smooth`: true

In [ ]:
check_pdal_available()

In [ ]:
# run_pipeline(
#     pipeline_json_path=pipeline_json_path,
#     input_laz=raw_las_path,
#     output_laz=normalized_las_path
# )

In [ ]:
raw_las        = load_laz(raw_las_path)
normalized_las = load_laz(normalized_las_path)

raw_df = pd.DataFrame({
    "x": np.array(raw_las.x).flatten(),
    "y": np.array(raw_las.y).flatten(),
    "z": np.array(raw_las.z).flatten(),
    "classification": np.array(raw_las.classification).flatten(),
    "return_number": np.array(raw_las.return_number).flatten(),
    "intensity": np.array(raw_las.intensity).flatten(),
})

normal_df = pd.DataFrame({
    "x": np.array(normalized_las.x).flatten(),
    "y": np.array(normalized_las.y).flatten(),
    "z": np.array(normalized_las.z).flatten(),
    "classification": np.array(normalized_las.classification).flatten(),
    "return_number": np.array(normalized_las.return_number).flatten(),
    "intensity": np.array(normalized_las.intensity).flatten(),
})

print(f"Raw classification counts:\n{raw_df.value_counts('classification')}")
print(f"Normalized classification counts:\n{normal_df.value_counts('classification')}")

In [ ]:
print(list(normalized_las.point_format.extra_dimension_names))

---

## Part 3 — CHM Derivation: DSM, DTM, and Canopy Height Model

Rasterizes the normalized point cloud into three raster products at 1m resolution:

1. **DSM** — max HeightAboveGround per pixel (top of canopy)
2. **DTM** — ground-classified returns only (bare earth)
3. **CHM** — DSM minus DTM (vegetation height above ground)

**Inputs:** `data/processed/socal_sespe_32611_normalized.laz`

**Outputs:** `outputs/rasters/socal_sespe_32611_{dsm,dtm,chm}.tif`

**Confirm before Part 4:**
- Plot the CHM — northern half should show higher values (denser vegetation), southern half lower
- Negative values are clamped to 0 in `compute_chm()`
- If raster is mostly zero or noisy, revisit ground classification in Part 2

**Resolution note:** Target is 1m. At ~2.2 pts/m² some cells may be empty — filled with NoData.

In [ ]:
rasterize_to_dsm(normalized_las_path, resolution=1.0, output_path=dsm_path)

In [ ]:
rasterize_to_dtm(normalized_las_path, resolution=1.0, output_path=dtm_path)

In [ ]:
compute_chm(dsm_path=dsm_path, dtm_path=dtm_path, output_path=chm_path)

In [ ]:
with rasterio.open(dsm_path) as src:
    dsm_data = src.read(1)

with rasterio.open(dtm_path) as src:
    dtm_data = src.read(1)

with rasterio.open(chm_path) as src:
    chm_data = src.read(1)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(dtm_data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.title('Digital Terrain Model (DTM)')
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(dsm_data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.title('Digital Surface Model (DSM)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(chm_data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.title('Canopy Height Model (CHM)')
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.show()

In [ ]:
chm_raster = rioxarray.open_rasterio(chm_path, masked=True).squeeze()
chm_3857 = chm_raster.rio.reproject("EPSG:3857")  # contextily needs Web Mercator

fig, ax = plt.subplots(figsize=(12, 5))
chm_3857.plot(ax=ax, alpha=0.8, cmap="gist_heat")
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
plt.show()

---

## Part 4 — Corridor Analysis: Buffering, Clipping, and Exceedance Detection

Applies the corridor analysis to the CHM from Part 3:

1. Load the CA transmission line centerline via ArcGIS REST API
2. Buffer by corridor width (default: 15m / 50ft)
3. Clip the CHM to the corridor buffer
4. Flag pixels exceeding the vegetation height threshold
5. Export final output artifacts

**Inputs:** CA transmission line centerline (CEC GIS API), `outputs/rasters/socal_sespe_32611_chm.tif`

**Outputs:**
- `outputs/rasters/socal_sespe_32611_chm_clipped.tif`
- `outputs/vector/flagged_trees.gpkg` — exceedance polygons with `height_m` attribute
- `outputs/tables/flagged_trees_summary.csv`

**Confirm before calling complete:**
- All three output files exist and are non-empty
- Flagged tree polygons fall within the corridor buffer (spatial sanity check)
- Run accuracy assessment in `validation/accuracy_assessment.py`

In [ ]:
url = (
    "https://services3.arcgis.com/bWPjFyq029ChCGur/arcgis/rest/services"
    "/Transmission_Line/FeatureServer/2/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

In [ ]:
clipped_transmission_lines = load_corridor_centerline(url, normalized_las_path)

In [ ]:
lines_with_buffer = buffer_corridor(clipped_transmission_lines, buffer_distance_m=15)

In [ ]:
clip_chm_to_corridor(
    chm_path=chm_path,
    corridor_gdf=lines_with_buffer,
    output_path=clipped_chm_path,
)

In [ ]:
threshold_df = threshold_exceedance(
    chm_clipped_path=clipped_chm_path,
    height_threshold_m=4.57  # 15ft — verify against FAC-003-4 for actual voltage class
)

In [ ]:
threshold_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
threshold_df.plot(column="height_m", cmap="Reds", legend=True, ax=ax)

In [ ]:
with rasterio.open(clipped_chm_path) as src:
    chm_clipped_data = src.read(1)

plt.figure(figsize=(10, 10))
plt.imshow(chm_clipped_data, cmap="viridis")
plt.colorbar(label="Height (m)")

In [ ]:
threshold_df.to_file(ROOT / "outputs" / "vector" / "flagged_trees.gpkg", driver="GPKG")

summary_df = threshold_df[["height_m"]].agg(
    count_exceedance_polygons=("height_m", "count"),
    mean_exceedance_height=("height_m", "mean")
)
summary_df.to_csv(ROOT / "outputs" / "tables" / "flagged_trees_summary.csv")

In [ ]:
threshold_df.height_m.describe()

In [ ]:
summary_df

---

## Part 5 — Accuracy Assessment: Thomas Fire Perimeter Validation

Uses the Thomas Fire burn perimeter as a spatial validation mask. This LiDAR was collected May–July 2018, ~4–6 months after the fire (December 2017). Within the perimeter, vegetation burned and CHM should be near zero. Outside, intact chaparral/foothill vegetation should produce meaningful canopy heights. The fire boundary is a field-free validation — the fire did the ground-truthing.

**Inputs:** Full-extent CHM (`outputs/rasters/socal_sespe_32611_chm.tif`), Thomas Fire perimeter (ArcGIS REST API, item `a2926cffbc854806befacbf85ebc9b95`)

**Outputs:** `outputs/tables/accuracy_summary.csv`

**Expected results:**
- Inside burn perimeter: mean CHM < 1m, >80% of pixels near zero
- Outside burn perimeter: mean CHM reflects intact vegetation structure (2–8m typical for chaparral/foothill)
- If inside and outside values are similar, diagnose ground classification or HAG normalization before treating flagged tree outputs as reliable

**Implement:** `validation/accuracy_assessment.py` → `assess_chm_accuracy(chm_path, fire_perimeter_path, output_path)`